In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# from pathlib import Path
# class Config:
#     train = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/Train.csv"
#     test = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/Test.csv"
#     DataDictionary = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/DataDictionary.csv"
#     SampleSubmission = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/SampleSubmission.csv"
#     root_dir = Path("/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/unido_rice_images")
#     target_columns = ['Count', 'Broken_Count', 'Long_Count', 'Medium_Count',
#        'Black_Count', 'Chalky_Count', 'Red_Count', 'Yellow_Count',
#        'Green_Count', 'WK_Length_Average', 'WK_Width_Average',
#        'WK_LW_Ratio_Average', 'Average_L', 'Average_a', 'Average_b']

#     IMG_SIZE = 512
#     # MODEL_NAME = 'vit_large_patch16_dinov3_qkvb'
#     MODEL_NAME = "vit_huge_patch14_224.mae"

# config = Config()

In [3]:
# train = pd.read_csv(config.train)
# test = pd.read_csv(config.test)
# train ['image_path'] = train['ID'].apply(lambda m: str(Config.root_dir/m) + ".png")
# test ['image_path'] = test['ID'].apply(lambda m: str(Config.root_dir/m) + ".png")

# train.head(5)

In [4]:
# from torch.utils.data import Dataset
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# import torch
# import cv2

# def get_embed_transforms():
#     return A.Compose([
#         A.Resize(config.IMG_SIZE, config.IMG_SIZE),
#         A.Normalize(mean=[0.485, 0.456, 0.406],
#                     std=[0.229, 0.224, 0.225]),
#         ToTensorV2()
#     ])



# class AfricanRiceDataset(Dataset):
    
#     def __init__(self, df, transform):
#         self.df = df
#         self.transform = transform
#         self.labels = df[config.target_columns].values.astype(np.float32)
    
#     def __len__(self):
#         return len(self.df)
    
#     def __getitem__(self, idx):
        
#         # Load image
#         path = self.df["image_path"].iloc[idx]
#         img = cv2.imread(path)
        
#         # Convert BGR to RGB
#         img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
#         # Split into left and right halves
#         h, w, _ = img.shape
#         mid = w // 2
#         left = img[:, :mid]
#         right = img[:, mid:]
        
#         # Apply transformations independently
#         left = self.transform(image=left)['image']
#         right = self.transform(image=right)['image']
        
#         # Get labels
#         label = torch.from_numpy(self.labels[idx])
        
#         return left, right, label

In [5]:
# import timm
# import torch
# import numpy as np
# from tqdm import tqdm

# device = "cuda" if torch.cuda.is_available() else "cpu"

# backbone = timm.create_model(
#     config.MODEL_NAME,
#     pretrained=True,
#     num_classes=0,
#     global_pool="avg"
# ).to(device)

# backbone.eval()

# import cv2

# def extract_and_save_embeddings(df, save_path, is_train=True):
#     feats = []
#     targets = []

#     transform = get_embed_transforms()

#     with torch.no_grad():
#         for _, row in tqdm(df.iterrows(), total=len(df)):
#             img = cv2.imread(row["image_path"])
#             img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

#             h, w, _ = img.shape
#             mid = w // 2
#             left = img[:, :mid]
#             right = img[:, mid:]

#             left = transform(image=left)["image"].unsqueeze(0).to(device)
#             right = transform(image=right)["image"].unsqueeze(0).to(device)

#             lf = backbone(left)
#             rf = backbone(right)

#             # ✅ concatenate exactly like your model
#             feat = torch.cat([lf, rf], dim=1)

#             feats.append(feat.cpu().numpy())

#             if is_train:
#                 targets.append(row[config.target_columns].values.astype(np.float32))

#     X = np.vstack(feats)

#     if is_train:
#         y = np.vstack(targets)
#         np.savez_compressed(save_path, X=X, y=y)
#     else:
#         np.savez_compressed(save_path, X=X)



In [6]:
# extract_and_save_embeddings(
#     train,
#     save_path="/kaggle/working/train_embeddings.npz",
#     is_train=True
# )



In [7]:
# extract_and_save_embeddings(
#     test,
#     save_path="/kaggle/working/test_embeddings.npz",
#     is_train=False
# )

In [8]:
# # ============================================================
# # FULL PIPELINE: CLS TOKEN EXTRACTION (LEFT + RIGHT)
# # MULTIPLE TIMM MODELS — SAVED SEPARATELY
# # ============================================================

# import numpy as np
# import pandas as pd
# import torch
# import timm
# import cv2
# from pathlib import Path
# from tqdm import tqdm

# import albumentations as A
# from albumentations.pytorch import ToTensorV2

# # ============================================================
# # CONFIG
# # ============================================================
# class Config:
#     train = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/Train.csv"
#     test = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/Test.csv"
#     root_dir = Path("/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/unido_rice_images")

#     target_columns = [
#         'Count', 'Broken_Count', 'Long_Count', 'Medium_Count',
#         'Black_Count', 'Chalky_Count', 'Red_Count', 'Yellow_Count',
#         'Green_Count', 'WK_Length_Average', 'WK_Width_Average',
#         'WK_LW_Ratio_Average', 'Average_L', 'Average_a', 'Average_b'
#     ]

#     IMG_SIZE = 512
#     SAVE_DIR = Path("/kaggle/working/cls_embeddings")

# config = Config()
# config.SAVE_DIR.mkdir(parents=True, exist_ok=True)

# # ============================================================
# # LOAD DATA
# # ============================================================
# train = pd.read_csv(config.train)
# test = pd.read_csv(config.test)

# train["image_path"] = train["ID"].apply(
#     lambda x: str(config.root_dir / x) + ".png"
# )
# test["image_path"] = test["ID"].apply(
#     lambda x: str(config.root_dir / x) + ".png"
# )

# # ============================================================
# # TRANSFORMS
# # ============================================================
# def get_embed_transforms():
#     return A.Compose([
#         A.Resize(config.IMG_SIZE, config.IMG_SIZE),
#         A.Normalize(
#             mean=[0.485, 0.456, 0.406],
#             std=[0.229, 0.224, 0.225]
#         ),
#         ToTensorV2()
#     ])

# # ============================================================
# # DEVICE
# # ============================================================
# device = "cuda" if torch.cuda.is_available() else "cpu"

# # ============================================================
# # MODELS TO EXPERIMENT WITH
# # ============================================================
# MODEL_NAMES = [
#     "vit_large_patch16_dinov3_qkvb",
#     "eva_giant_patch14_224.clip_ft_in1k",
#     "eva_large_patch14_336.in22k_ft_in1k",
#     "beitv2_large_patch16_224.in1k",
#     "deit3_large_patch16_224",
#     "convnextv2_large.fcmae",
#     "swinv2_large_window12_192_22k",
#     "coatnet_3_rw_224",
#     "vit_huge_patch14_224.mae",
#     "maxvit_large_tf_512"
# ]

# # ============================================================
# # BACKBONE FACTORY (NO GLOBAL POOL → KEEP TOKENS)
# # ============================================================
# def create_backbone(model_name):
#     model = timm.create_model(
#         model_name,
#         pretrained=True,
#         num_classes=0,
#         global_pool=""
#     )
#     model.eval()
#     return model.to(device)

# # ============================================================
# # CLS EXTRACTION
# # ============================================================
# def extract_cls(model, x):
#     out = model(x)      # (B, N, D)
#     return out[:, 0]    # CLS token

# # ============================================================
# # EXTRACTION + SAVE
# # ============================================================
# def extract_and_save_cls_embeddings(model_name, df, is_train=True):
#     print(f"\n🚀 Extracting CLS tokens from: {model_name}")

#     model = create_backbone(model_name)
#     transform = get_embed_transforms()

#     feats = []
#     targets = []

#     with torch.no_grad():
#         for _, row in tqdm(df.iterrows(), total=len(df)):
#             img = cv2.imread(row["image_path"])
#             img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

#             h, w, _ = img.shape
#             mid = w // 2

#             left = img[:, :mid]
#             right = img[:, mid:]

#             left = transform(image=left)["image"].unsqueeze(0).to(device)
#             right = transform(image=right)["image"].unsqueeze(0).to(device)

#             lf = extract_cls(model, left)
#             rf = extract_cls(model, right)

#             feat = torch.cat([lf, rf], dim=1)
#             feats.append(feat.cpu().numpy())

#             if is_train:
#                 targets.append(
#                     row[config.target_columns].values.astype(np.float32)
#                 )

#     X = np.vstack(feats)

#     save_name = model_name.replace(".", "_")
#     save_path = config.SAVE_DIR / f"{save_name}_cls.npz"

#     if is_train:
#         y = np.vstack(targets)
#         np.savez_compressed(save_path, X=X, y=y)
#     else:
#         np.savez_compressed(save_path, X=X)

#     print(f"✅ Saved: {save_path}")

# # ============================================================
# # RUN EXTRACTION
# # ============================================================

# # TRAIN
# for model_name in MODEL_NAMES:
#     extract_and_save_cls_embeddings(
#         model_name=model_name,
#         df=train,
#         is_train=True
#     )

# # TEST
# for model_name in MODEL_NAMES:
#     extract_and_save_cls_embeddings(
#         model_name=model_name,
#         df=test,
#         is_train=False
#     )

# print("\n🎯 ALL CLS EMBEDDINGS EXTRACTED SUCCESSFULLY")


In [9]:
# ============================================================
# AVG-POOLED FEATURE EXTRACTION (LEFT + RIGHT)
# UNIVERSAL TIMM VERSION
# ============================================================

import numpy as np
import pandas as pd
import torch
import timm
import cv2
from pathlib import Path
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

# ============================================================
# CONFIG
# ============================================================
class Config:
    train = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/Train.csv"
    test = "/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/Test.csv"
    root_dir = Path("/kaggle/input/dataset-africarice/Unido_AfricaRice_Challenge/unido_rice_images")

    target_columns = [
        'Count', 'Broken_Count', 'Long_Count', 'Medium_Count',
        'Black_Count', 'Chalky_Count', 'Red_Count', 'Yellow_Count',
        'Green_Count', 'WK_Length_Average', 'WK_Width_Average',
        'WK_LW_Ratio_Average', 'Average_L', 'Average_a', 'Average_b'
    ]

    SAVE_DIR = Path("/kaggle/working/avgpooled_embeddings")

config = Config()
config.SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================
train = pd.read_csv(config.train)
test = pd.read_csv(config.test)

train["image_path"] = train["ID"].apply(lambda x: str(config.root_dir / x) + ".png")
test["image_path"] = test["ID"].apply(lambda x: str(config.root_dir / x) + ".png")

# ============================================================
# DEVICE
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# MODELS
# ============================================================
MODEL_NAMES = [
    # "vit_large_patch16_dinov3_qkvb",
    # "eva_giant_patch14_224.clip_ft_in1k",
    # "eva_large_patch14_336.in22k_ft_in1k",
    # "beitv2_large_patch16_224.in1k",
    # "deit3_large_patch16_224",
    # "convnextv2_large.fcmae",
    # "swinv2_large_window12_192_22k",
    # "coatnet_3_rw_224",
    "vit_huge_patch14_224.mae",
    # "maxvit_large_tf_512"
]

# ============================================================
# BACKBONE FACTORY
# ============================================================
def create_backbone(model_name):
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=0,
        global_pool=""  # keep tokens for transformer models
    )
    model.eval()
    return model.to(device)

# ============================================================
# TRANSFORMS (MODEL-AWARE SIZE)
# ============================================================
def get_embed_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])

# ============================================================
# AVG-POOLED EXTRACTION
# ============================================================
def extract_avgpooled(model, x):
    out = model(x)  # shape: (B, N, D) for transformers, (B, D) for CNNs
    if out.ndim == 3:  # transformer sequence
        return out.mean(dim=1)  # average pool across tokens
    return out  # CNN already pooled

# ============================================================
# EXTRACTION + SAVE
# ============================================================
def extract_and_save_avgpooled_embeddings(model_name, df, is_train=True):
    print(f"\n🚀 Extracting AVG-POOLED features from: {model_name}")

    model = create_backbone(model_name)

    # Model-native input size
    input_size = model.default_cfg["input_size"][1]
    transform = get_embed_transforms(input_size)

    feats = []
    targets = []

    with torch.no_grad():
        for _, row in tqdm(df.iterrows(), total=len(df)):
            img = cv2.imread(row["image_path"])
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            h, w, _ = img.shape
            mid = w // 2
            left = img[:, :mid]
            right = img[:, mid:]

            left = transform(image=left)["image"].unsqueeze(0).to(device)
            right = transform(image=right)["image"].unsqueeze(0).to(device)

            lf = extract_avgpooled(model, left)
            rf = extract_avgpooled(model, right)

            feat = torch.cat([lf, rf], dim=1)
            feats.append(feat.cpu().numpy())

            if is_train:
                targets.append(row[config.target_columns].values.astype(np.float32))

    X = np.vstack(feats)
    save_name = model_name.replace(".", "_")
    

    if is_train:
        save_path = config.SAVE_DIR / f"{save_name}_train_avgpooled.npz"
        y = np.vstack(targets)
        np.savez_compressed(save_path, X=X, y=y)
    else:
        save_path = config.SAVE_DIR / f"{save_name}_test_avgpooled.npz"
        np.savez_compressed(save_path, X=X)

    print(f"✅ Saved: {save_path}")

# ============================================================
# RUN EXTRACTION
# ============================================================
# TRAIN
for model_name in MODEL_NAMES:
    extract_and_save_avgpooled_embeddings(
        model_name=model_name,
        df=train,
        is_train=True
    )

# TEST
for model_name in MODEL_NAMES:
    extract_and_save_avgpooled_embeddings(
        model_name=model_name,
        df=test,
        is_train=False
    )

print("\n🎯 ALL AVG-POOLED EMBEDDINGS EXTRACTED SUCCESSFULLY")


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 


🚀 Extracting AVG-POOLED features from: vit_huge_patch14_224.mae


model.safetensors:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

100%|██████████| 938/938 [06:47<00:00,  2.30it/s]


✅ Saved: /kaggle/working/avgpooled_embeddings/vit_huge_patch14_224_mae_train_avgpooled.npz

🚀 Extracting AVG-POOLED features from: vit_huge_patch14_224.mae


100%|██████████| 403/403 [02:51<00:00,  2.35it/s]


✅ Saved: /kaggle/working/avgpooled_embeddings/vit_huge_patch14_224_mae_test_avgpooled.npz

🎯 ALL AVG-POOLED EMBEDDINGS EXTRACTED SUCCESSFULLY
